In [1]:
import numpy as np
import os

In [2]:
def load_dataset(csv_path: str):
    df = pd.read_csv(csv_path, low_memory=False)

    wave_cols = [c for c in df.columns if str(c).startswith("Wave_")]
    if len(wave_cols) < 10:
        raise ValueError(f"В файле {csv_path} не найдены Wave_* колонки")

    def parse_wave(c):
        return float(str(c).split("_", 1)[1])

    wave_axis = np.array([parse_wave(c) for c in wave_cols], dtype=float)
    order = np.argsort(wave_axis)
    wave_axis = wave_axis[order]
    wave_cols = [wave_cols[i] for i in order]

    #keep_meta = [c for c in ["position", "class", "group", "place", "X", "Y"] if c in df.columns]
    #df = df[keep_meta + wave_cols].copy()

    for c in wave_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    if "X" in df.columns:
        df["X"] = pd.to_numeric(df["X"], errors="coerce")
    if "Y" in df.columns:
        df["Y"] = pd.to_numeric(df["Y"], errors="coerce")

    return df, wave_axis, wave_cols

In [3]:
def build_windows_for_axis(wave_axis: np.ndarray):
    lo, hi = wave_axis.min(), wave_axis.max()

    if hi < 2200:
        return [
            (990, 1010, "W1_990-1010"),
            (1025, 1065, "W2_1025-1065"),
            (1110, 1160, "W3_1110-1160"),
            (1230, 1310, "W4_1230-1310"),
            (1430, 1470, "W5_1430-1470"),
            (1620, 1700, "W6_1620-1700"),
            (1850, 2000, "W7_1850-2000_tail"),
        ]

    return [
        (2458, 2600, "W1_2458-2600"),
        (2800, 2860, "W2_2800-2860"),
        (2860, 2935, "W3_2860-2935"),
        (2935, 2985, "W4_2935-2985"),
        (2985, 3050, "W5_2985-3050"),
        (3050, 3150, "W6_3050-3150"),
        (3150, 3288, "W7_3150-3288_tail"),
    ]



In [4]:
from scipy.signal import find_peaks

In [5]:
def build_windows_for_axis_by_peaks(
    df,
    wave_axis,
    wave_cols,
    n_peaks=10,
    half_width=50.0,
    min_distance=None,
):
    Xmat = df[wave_cols].to_numpy(dtype=float)
    spectrum = pd.Series(np.nanmean(Xmat, axis=0)).ffill().bfill().to_numpy()

    if min_distance is None:
        min_distance = len(wave_axis) // 50

    peaks, props = find_peaks(spectrum, distance=min_distance, prominence=0)

    if len(peaks) == 0:
        raise ValueError("No peaks found.")

    prominences = props["prominences"]
    top_idx = np.argsort(prominences)[-n_peaks:]
    top_peaks = peaks[top_idx]
    top_prom = prominences[top_idx]

    windows = [
        (
            float(wave_axis[p] - half_width),
            float(wave_axis[p] + half_width),
            f"P{i}_{wave_axis[p]:.1f}"
        )
        for i, p in enumerate(top_peaks, start=1)
    ]

    return windows, top_peaks, top_prom, spectrum

In [6]:
def print_windows(windows):
    for i, (lo, hi, name) in enumerate(windows, 1):
        center = (lo + hi) / 2
        width = hi - lo
        print(f"{i:2d}. {name:15}  center={center:8.2f}  [{lo:8.2f}, {hi:8.2f}]  width={width:.2f}")

In [7]:
def build_window_features(df, wave_axis, wave_cols, prefix=""):
    Xmat = df[wave_cols].to_numpy(dtype=float)

    windows, top_peaks, top_prom, mean_spectrum = build_windows_for_axis_by_peaks(
        df=df,
        wave_axis=wave_axis,
        wave_cols=wave_cols,
        n_peaks=10,
        half_width=50,
    )

    print_windows(windows)

    feat = {}

    # Глобальные статистики
    feat[f"{prefix}Global_min"] = np.min(Xmat, axis=1)
    feat[f"{prefix}Global_max"] = np.max(Xmat, axis=1)
    feat[f"{prefix}Global_range"] = feat[f"{prefix}Global_max"] - feat[f"{prefix}Global_min"]
    feat[f"{prefix}Global_mean"] = np.mean(Xmat, axis=1)
    feat[f"{prefix}Global_std"] = np.std(Xmat, axis=1)
    feat[f"{prefix}Global_median"] = np.median(Xmat, axis=1)

    window_names = []
    for lo, hi, name in windows:
        mask = (wave_axis >= lo) & (wave_axis <= hi)
        Xm = Xmat[:, mask]

        nm = f"{prefix}{name}"
        window_names.append(nm)

        feat[nm] = np.trapezoid(Xm, x=wave_axis[mask], axis=1)

        feat[f"{prefix}Min_{name}"] = np.min(Xm, axis=1)
        feat[f"{prefix}Max_{name}"] = np.max(Xm, axis=1)
        feat[f"{prefix}Range_{name}"] = feat[f"{prefix}Max_{name}"] - feat[f"{prefix}Min_{name}"]

        feat[f"{prefix}Mean_{name}"] = np.mean(Xm, axis=1)
        feat[f"{prefix}Std_{name}"] = np.std(Xm, axis=1)
        feat[f"{prefix}Median_{name}"] = np.median(Xm, axis=1)

    F = pd.DataFrame(feat).ffill().bfill()
    F = F.fillna(F.mean(numeric_only=True))

    eps = 1e-12
    W = {k: F[k].to_numpy() for k in window_names}
    sumW = np.sum(np.column_stack([W[k] for k in window_names]), axis=1) + eps

    for k in window_names:
        F[f"{prefix}Frac_{k}"] = (W[k] + eps) / sumW

    F[f"{prefix}SquareUP"] = np.trapezoid(Xmat, axis=1)

    return F

In [8]:
def build_derivative_features(df, wave_axis, wave_cols, prefix="D_"):
    Xmat = df[wave_cols].to_numpy(dtype=float)
    dX = np.gradient(Xmat, wave_axis, axis=1)

    windows, top_peaks, top_prom, mean_spectrum = build_windows_for_axis_by_peaks(
        df=df,
        wave_axis=wave_axis,
        wave_cols=wave_cols,
        n_peaks=10,
        half_width=50,
    )
    feat = {}

    feat[f"{prefix}Global_min"] = np.min(dX, axis=1)
    feat[f"{prefix}Global_max"] = np.max(dX, axis=1)
    feat[f"{prefix}Global_range"] = feat[f"{prefix}Global_max"] - feat[f"{prefix}Global_min"]

    feat[f"{prefix}Global_mean"] = np.mean(dX, axis=1)
    feat[f"{prefix}Global_std"] = np.std(dX, axis=1)
    feat[f"{prefix}Global_median"] = np.median(dX, axis=1)

    for lo, hi, name in windows:
        mask = (wave_axis >= lo) & (wave_axis <= hi)
        Xm = dX[:, mask]

        feat[f"{prefix}{name}"] = np.trapezoid(np.abs(Xm), x=wave_axis[mask], axis=1)
        feat[f"{prefix}{name}_min"] = np.min(Xm, axis=1)
        feat[f"{prefix}{name}_max"] = np.max(Xm, axis=1)
        feat[f"{prefix}{name}_range"] = np.max(Xm, axis=1) - np.min(Xm, axis=1)
        feat[f"{prefix}{name}_mean"] = np.mean(Xm, axis=1)
        feat[f"{prefix}{name}_std"] = np.std(Xm, axis=1)
        feat[f"{prefix}{name}_median"] = np.median(Xm, axis=1)

    F = pd.DataFrame(feat).ffill().bfill()
    F = F.fillna(F.mean(numeric_only=True))
    return F

In [9]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import normalize
from pybaselines import Baseline


def preprocess_wave_features(df):
    #Baseline correction + normalization для колонок Wave_
    wave_cols = [c for c in df.columns if c.startswith("Wave_")]
    X = df[wave_cols].to_numpy(dtype=float)

    baseline_fitter = Baseline()

    X_corr = np.zeros_like(X)

    for i, spectrum in enumerate(X):
        baseline, _ = baseline_fitter.asls(spectrum, lam=1e5, p=0.01)
        X_corr[i] = spectrum - baseline

    X_norm = normalize(X_corr, norm="l2")

    df[wave_cols] = X_norm
    return df

In [10]:
def produce_feat(path, output):
    df, wave_axis, wave_cols = load_dataset(path)

    df = preprocess_wave_features(df)

    new_features = build_window_features(df, wave_axis, wave_cols)
    df = pd.concat([df, new_features], axis=1)

    new_features = build_derivative_features(df, wave_axis, wave_cols)
    df = pd.concat([df, new_features], axis=1)

    df.to_csv(output, index=False)

In [11]:
os.chdir("/")

if __name__ == "__main__":

    produce_feat('mergedData/data_1500.csv', 'mergedData/data_feat_1500_std.csv')

    produce_feat('mergedData/data_2900.csv', 'mergedData/data_feat_2900_std.csv')

 1. P1_1785.8        center= 1785.80  [ 1735.80,  1835.80]  width=100.00
 2. P2_1002.5        center= 1002.50  [  952.50,  1052.50]  width=100.00
 3. P3_1660.2        center= 1660.20  [ 1610.20,  1710.20]  width=100.00
 4. P4_1417.1        center= 1417.10  [ 1367.10,  1467.10]  width=100.00
 5. P5_1170.1        center= 1170.10  [ 1120.10,  1220.10]  width=100.00
 6. P6_1460.8        center= 1460.80  [ 1410.80,  1510.80]  width=100.00
 7. P7_1061.3        center= 1061.30  [ 1011.30,  1111.30]  width=100.00
 8. P8_1131.8        center= 1131.80  [ 1081.80,  1181.80]  width=100.00
 9. P9_1294.5        center= 1294.50  [ 1244.50,  1344.50]  width=100.00
10. P10_1438.5       center= 1438.50  [ 1388.50,  1488.50]  width=100.00
 1. P1_3043.3        center= 3043.30  [ 2993.30,  3093.30]  width=100.00
 2. P2_3164.5        center= 3164.50  [ 3114.50,  3214.50]  width=100.00
 3. P3_2959.0        center= 2959.00  [ 2909.00,  3009.00]  width=100.00
 4. P4_2581.5        center= 2581.50  [ 2531.50,  2

In [12]:
os.chdir("/mergedData")
df = pd.read_csv("mergedData/data_feat_1500_std.csv")

C:\Users\Zepspel\AppData\Local\Temp\ipykernel_7336\3097295871.py:2: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data_feat_1500_std.csv")


In [13]:
print(df.head())

  position    class group   nm  center  obj  power during  acc    map  ...  \
0   cortex  control     1  633    1500  100    100     1s    5  35x15  ...   
1   cortex  control     1  633    1500  100    100     1s    5  35x15  ...   
2   cortex  control     1  633    1500  100    100     1s    5  35x15  ...   
3   cortex  control     1  633    1500  100    100     1s    5  35x15  ...   
4   cortex  control     1  633    1500  100    100     1s    5  35x15  ...   

   D_P9_1294.5_mean D_P9_1294.5_std  D_P9_1294.5_median  D_P10_1438.5  \
0          0.000347        0.008567            0.000517      0.649304   
1         -0.000046        0.009684            0.000565      0.818000   
2         -0.000261        0.009061           -0.000165      0.736683   
3         -0.000124        0.009159            0.000161      0.721774   
4         -0.000023        0.008315           -0.000051      0.719119   

   D_P10_1438.5_min  D_P10_1438.5_max  D_P10_1438.5_range  D_P10_1438.5_mean  \
0         -0